In [0]:
from datetime import datetime
container_name = dbutils.widgets.get("container_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
folder_name = dbutils.widgets.get("folder_name")
archive = dbutils.widgets.get("archive")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
storage_account_key = dbutils.secrets.get(scope="amazon", key="storage_account_key")
# base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
folder_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{folder_name}"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
archive_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{archive}"
##dbutils.fs.put(f"{folder_path}/{datetime.now()}.json", json.dumps(data), overwrite=True)

In [0]:
items = dbutils.fs.ls(f'{folder_path}')
loaded_files = [item.name for item in items]
loaded_files

In [0]:
df = spark.sql(f'SELECT DISTINCT(file_name) FROM {catalog}.{schema}.amazon_orders_silver')

In [0]:
filenames_in_raw_table = [row.file_name for row in df.collect()]
filenames_in_raw_table

# COMMAND ----------

loaded_files_set = set(loaded_files)
filenames_in_raw_table_set = set(filenames_in_raw_table)

# COMMAND ----------

files_to_move = list(loaded_files_set.intersection(filenames_in_raw_table_set))

for file_name in files_to_move:
    source_path = f"{folder_path}/{file_name}"
    destination_path = f"{archive_path}/{file_name}"
    
    try:
        dbutils.fs.ls(source_path)
        dbutils.fs.mv(source_path, destination_path)
        print(f"Moved {file_name} to {destination_path}")
    except Exception as e:
        print(f"Source path does not exist: {source_path}")